IMPORTS

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords
import warnings
warnings.filterwarnings("ignore")

nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [23]:
data = r"C:\Users\Admin\Desktop\JAYESH\Projects\Email spam\dataset\spam.csv"
df = pd.read_csv(data,encoding='latin-1')

In [24]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [25]:
df.drop(columns=['Unnamed: 2','Unnamed: 3','Unnamed: 4'],inplace=True)
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [26]:
df.rename(columns={'v1':'target','v2':'text'},inplace=True)
df.head()

,target,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


PREprocessing

In [27]:
from sklearn.preprocessing import LabelEncoder

lb = LabelEncoder()
df['target'] = lb.fit_transform(df['target'])
df.head()

,target,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [28]:
df.duplicated().sum()

403

In [29]:
len(df)

5572

In [30]:
df = df.drop_duplicates(keep='first')
df.tail()

,target,text
5567,1,This is the 2nd time we have tried 2 contact u...
5568,0,Will Ì_ b going to esplanade fr home?
5569,0,"Pity, * was in mood for that. So...any other s..."
5570,0,The guy did some bitching but I acted like i'd...
5571,0,Rofl. Its true to its name


Feature Engineering

In [31]:
from nltk.stem import WordNetLemmatizer
import string

wl = WordNetLemmatizer()

In [32]:
# lowercase transformation and text preprocessing function 

def transform_text(text):
    text = text.lower()
    text = nltk.word_tokenize(text)

    # Remove spcl char
    y=[]
    for i in text:
        if i.isalnum():
            y.append(i)
    
    # Removing stopwords and punctuations
    text = y[:]
    y.clear()

    # Loop through the tokens and remove stopwords and punctuations
    for i in text:
        if i not in stopwords.words('english') and i not in string.punctuation:
            y.append(i)
    
    #Lemmatizing
    text=y[:]
    y.clear()
    for i in text:
        y.append(wl.lemmatize(i))

    # Join the processed tokens back into a single string
    return "".join(y)

In [33]:
df['transformed_text'] = df['text'].apply(transform_text)
df.head()

,target,text,transformed_text
0,0,"Go until jurong point, crazy.. Available only ...",gojurongpointcrazyavailablebugisngreatworldlae...
1,0,Ok lar... Joking wif u oni...,oklarjokingwifuoni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,freeentry2wklycompwinfacupfinaltkts21stmaytext...
3,0,U dun say so early hor... U c already then say...,udunsayearlyhorucalreadysay
4,0,"Nah I don't think he goes to usf, he lives aro...",nahthinkgousflifearoundthough


In [34]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
tfidf = TfidfVectorizer(max_features=5000)

In [35]:
x = tfidf.fit_transform(df['transformed_text']).toarray()
y = df['target'].values

TRAIN TEST SPLIT

In [36]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.25,random_state=28,stratify=y)

In [37]:
# from imblearn.over_sampling import SMOTE

# smote = SMOTE(random_state=66)

# x_train,y_train = smote.fit_resample(x_train,y_train)

MODEL TRAINING

In [38]:
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from xgboost import XGBClassifier
from sklearn.linear_model import RidgeClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.linear_model import PassiveAggressiveClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [ ]:
# Linear Models
lr = LogisticRegression(random_state=42,class_weight='balanced')
ridge = RidgeClassifier(random_state=42)
sgd = SGDClassifier(loss='log_loss', random_state=42)
pa = PassiveAggressiveClassifier(random_state=42)

# Naive Bayes
bnb = MultinomialNB()

# Tree-based Models
dt = DecisionTreeClassifier(max_depth=5, random_state=42)

rf = RandomForestClassifier(n_estimators=100, random_state=42,class_weight='balanced')
et = ExtraTreesClassifier(n_estimators=100, random_state=42)

gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
ada = AdaBoostClassifier(n_estimators=100, random_state=42)

# SVM
svc = SVC(kernel='sigmoid',random_state=42)

# KNN
knn = KNeighborsClassifier()

# XGBoost
xgb = XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='logloss')

In [ ]:
clfs = [
    ('Logistic Regression', lr),
    ('Ridge', ridge),
    ('SGD', sgd),
    ('Passive Aggressive', pa),
    ('Naive Bayes', bnb),
    ('Decision Tree', dt),
    ('Random Forest', rf),
    ('Extra Trees', et),
    ('Gradient Boosting', gb),
    ('AdaBoost', ada),
    ('SVC', svc),
    ('KNN', knn),
    ('XGBoost', xgb),
]

MODEL EVALUATION

In [41]:
from sklearn.metrics import accuracy_score,precision_score
from sklearn.metrics import recall_score, f1_score

def train_classifier(clfs,x_train,y_train,x_test,y_test):
    clfs.fit(x_train,y_train)
    y_pred = clfs.predict(x_test)
    acc = accuracy_score(y_test,y_pred)
    precision = precision_score(y_test,y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    return acc,precision,rec,f1

In [42]:
accuracy_scores = []
precision_scores = []
recall_scores = []
f1_scores = []

for name, model in clfs:
    acc, prec,rec,f1 = train_classifier(model, x_train, y_train, x_test, y_test)
    
    accuracy_scores.append(acc)
    precision_scores.append(prec)
    recall_scores.append(rec)
    f1_scores.append(f1)
    
    print(f"For {name}")
    print(f"Accuracy: {acc}")
    print(f"Precision: {prec}")
    print(f"Recall: {rec}")
    print(f"F1: {f1}")
    print()

For Logistic Regression
Accuracy: 0.8808971384377416
Precision: 1.0
Recall: 0.05521472392638037
F1: 0.10465116279069768

For Ridge
Accuracy: 0.8808971384377416
Precision: 1.0
Recall: 0.05521472392638037
F1: 0.10465116279069768



c:\Users\Admin\Desktop\JAYESH\Projects\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


For SGD
Accuracy: 0.8739365815931941
Precision: 0.0
Recall: 0.0
F1: 0.0

For Passive Aggressive
Accuracy: 0.8808971384377416
Precision: 1.0
Recall: 0.05521472392638037
F1: 0.10465116279069768

For Naive Bayes
Accuracy: 0.8739365815931941
Precision: 0.0
Recall: 0.0
F1: 0.0



c:\Users\Admin\Desktop\JAYESH\Projects\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Admin\Desktop\JAYESH\Projects\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


For Decision Tree
Accuracy: 0.8739365815931941
Precision: 0.0
Recall: 0.0
F1: 0.0

For Random Forest
Accuracy: 0.8808971384377416
Precision: 1.0
Recall: 0.05521472392638037
F1: 0.10465116279069768

For Extra Trees
Accuracy: 0.8808971384377416
Precision: 1.0
Recall: 0.05521472392638037
F1: 0.10465116279069768



...

c:\Users\Admin\Desktop\JAYESH\Projects\venv\lib\site-packages\xgboost\training.py:200: UserWarning: [21:19:39] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
c:\Users\Admin\Desktop\JAYESH\Projects\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


For XGBoost
Accuracy: 0.8739365815931941
Precision: 0.0
Recall: 0.0
F1: 0.0

[LightGBM] [Warning] There are no meaningful features which satisfy the provided configuration. Decreasing Dataset parameters min_data_in_bin or min_data_in_leaf and re-constructing Dataset might resolve this warning.
[LightGBM] [Info] Number of positive: 490, number of negative: 3386
[LightGBM] [Info] Total Bins 0
[LightGBM] [Info] Number of data points in the train set: 3876, number of used features: 0
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.126419 -> initscore=-1.932999
[LightGBM] [Info] Start training from score -1.932999
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] Stopped training because there are no more leave

c:\Users\Admin\Desktop\JAYESH\Projects\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\Admin\Desktop\JAYESH\Projects\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


For CatBoost
Accuracy: 0.8739365815931941
Precision: 0.0
Recall: 0.0
F1: 0.0



c:\Users\Admin\Desktop\JAYESH\Projects\venv\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
